# Data Apps with Streamlit

**Streamlit** turns a Python script into an interactive web app: widgets (select boxes, sliders), charts and tables, without HTML or JavaScript. Data teams use it to share analyses, prototypes and internal tools in hours instead of weeks.

**After this lesson you can:**

1. Explain Streamlit's execution model (the script reruns on every interaction).
2. Write an app with widgets, layout, charts and caching.
3. Test an app automatically with `AppTest`, without a browser.
4. Run the app as a server and deploy it.

**Data:** NYC yellow taxi trips, January 2024, pre-aggregated to zone × hour.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "tools"))
import datasets                                   # course helper: downloads pinned public datasets

plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 100})

import subprocess
import tempfile
import time
import urllib.request

import streamlit as st

APP_DIR = Path(tempfile.mkdtemp())
print("Streamlit", st.__version__)

Streamlit 1.67.0


## 1. Prepare the Data

A data app should load **small, pre-aggregated** data, so it starts fast. The cell aggregates 3 million trips to one row per pickup zone, weekday and hour, and saves it as Parquet next to the app.

In [2]:
taxi = pd.read_parquet(datasets.taxi("2024-01"), columns=["tpep_pickup_datetime", "PULocationID", "fare_amount", "tip_amount"])
taxi = taxi[taxi["tpep_pickup_datetime"].between("2024-01-01", "2024-02-01", inclusive="left") & taxi["fare_amount"].between(3, 500)]
zones = pd.read_csv(datasets.fetch("taxi_zone_lookup.csv"))
agg = (taxi.assign(weekday=taxi["tpep_pickup_datetime"].dt.day_name(), hour=taxi["tpep_pickup_datetime"].dt.hour)
       .groupby(["PULocationID", "weekday", "hour"])
       .agg(trips=("fare_amount", "size"), fare_sum=("fare_amount", "sum"), tip_sum=("tip_amount", "sum")).reset_index()
       .merge(zones[["LocationID", "Borough", "Zone"]], left_on="PULocationID", right_on="LocationID").drop(columns="LocationID")
       .dropna(subset=["Borough", "Zone"]))                      # pandas reads the "N/A" borough of zone 265 as missing
agg.to_parquet(APP_DIR / "zone_hour.parquet", index=False)
print(f"{len(taxi):,} trips -> {len(agg):,} aggregated rows, {(APP_DIR / 'zone_hour.parquet').stat().st_size / 1024:.0f} KiB")

2,924,610 trips -> 26,112 aggregated rows, 279 KiB


## 2. The Execution Model

A Streamlit app is a script. On every interaction (a new selection, a slider move), Streamlit **runs the whole script again from the top**, with the new widget values. Two tools keep this fast and stateful:

| Tool | Purpose |
| :--- | :--- |
| `@st.cache_data` | cache the result of a function (for example loading data) across reruns |
| `st.session_state` | keep values (a counter, a login, a chat history) across reruns |

## 3. Write the App

In [3]:
APP = r"""
from pathlib import Path

import pandas as pd
import plotly.express as px
import streamlit as st

st.set_page_config(page_title="Taxi Zone Explorer", layout="wide")
DATA = Path(__file__).parent / "zone_hour.parquet"
DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

@st.cache_data
def load() -> pd.DataFrame:
    return pd.read_parquet(DATA)

df = load()
st.title("NYC Taxi Zone Explorer - January 2024")

with st.sidebar:
    borough = st.selectbox("Borough", sorted(df["Borough"].unique()), index=sorted(df["Borough"].unique()).index("Manhattan"))
    zone_options = sorted(df.loc[df["Borough"] == borough, "Zone"].unique())
    zone = st.selectbox("Pickup zone", zone_options)
    days = st.multiselect("Weekdays", DAYS, default=DAYS)

sel = df[(df["Zone"] == zone) & df["weekday"].isin(days)]
trips = int(sel["trips"].sum())
c1, c2, c3 = st.columns(3)
c1.metric("Trips", f"{trips:,}")
c2.metric("Average fare", f"${sel['fare_sum'].sum() / max(trips, 1):.2f}")
c3.metric("Tips per fare dollar", f"{100 * sel['tip_sum'].sum() / max(sel['fare_sum'].sum(), 1):.1f}%")

by_hour = sel.groupby("hour", as_index=False)["trips"].sum()
st.plotly_chart(px.bar(by_hour, x="hour", y="trips", title=f"Pickups by hour in {zone}"), width="stretch")
st.caption("Source: NYC TLC trip records. Fares $3-$500.")
"""
(APP_DIR / "app.py").write_text(APP, encoding="utf-8")
print(f"app.py: {len(APP.splitlines())} lines")

app.py: 34 lines


## 4. Test the App without a Browser

`streamlit.testing.v1.AppTest` runs the script in the test process, exposes all widgets and outputs, and lets you **simulate interactions**. This is how you test apps in CI (M34).

In [4]:
from streamlit import logger as st_logger
from streamlit.testing.v1 import AppTest

st_logger.set_log_level("error")                 # hide Streamlit's info and warning messages in the notebook

at = AppTest.from_file(str(APP_DIR / "app.py"), default_timeout=60).run()
assert not at.exception, at.exception
print("title   :", at.title[0].value)
print("widgets :", [sb.label for sb in at.selectbox], [ms.label for ms in at.multiselect])
print("metrics :", {m.label: m.value for m in at.metric})

at.selectbox[0].select("Queens").run()                       # choose a borough ...
at.selectbox[1].select("JFK Airport").run()                  # ... and a zone
print("\nafter selecting JFK Airport:", {m.label: m.value for m in at.metric})
at.multiselect[0].set_value(["Saturday", "Sunday"]).run()    # weekends only
print("weekends only            :", {m.label: m.value for m in at.metric})
assert not at.exception

title   : NYC Taxi Zone Explorer - January 2024
widgets : ['Borough', 'Pickup zone'] ['Weekdays']
metrics : {'Trips': '3,490', 'Average fare': '$17.30', 'Tips per fare dollar': '14.1%'}

after selecting JFK Airport: {'Trips': '141,070', 'Average fare': '$62.43', 'Tips per fare dollar': '14.6%'}
weekends only            : {'Trips': '36,256', 'Average fare': '$62.19', 'Tips per fare dollar': '14.9%'}


In [5]:
check = agg[(agg["Zone"] == "JFK Airport") & agg["weekday"].isin(["Saturday", "Sunday"])]
expected = f"{int(check['trips'].sum()):,}"
print("app shows:", at.metric[0].value, "| computed directly:", expected, "| match:", at.metric[0].value == expected)

app shows: 36,256 | computed directly: 36,256 | match: True


The test checks the app's number against an independent computation, so a bug in the app's filter would fail the test.

## 5. Run the App as a Server

`streamlit run app.py` starts a web server (port 8501 by default). In a terminal you would open the URL in a browser; here the cell starts the server in the background, checks its health endpoint, and stops it.

In [6]:
port = 8599
proc = subprocess.Popen([sys.executable, "-m", "streamlit", "run", str(APP_DIR / "app.py"), "--server.headless=true",
                         f"--server.port={port}", "--browser.gatherUsageStats=false"],
                        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
try:
    for _ in range(60):
        try:
            with urllib.request.urlopen(f"http://localhost:{port}/_stcore/health", timeout=1) as r:
                print("health endpoint:", r.status, r.read().decode())
                break
        except Exception:
            time.sleep(0.5)
    with urllib.request.urlopen(f"http://localhost:{port}/", timeout=5) as r:
        page = r.read().decode()
    print("main page served:", r.status, "| contains the Streamlit app shell:", "streamlit" in page.lower())
finally:
    proc.terminate()
    proc.wait(timeout=10)
print("server stopped")

health endpoint: 200 ok
main page served: 200 | contains the Streamlit app shell: True


server stopped


## 6. Deploy and Compare

| Option | Cost | Notes |
| :--- | :--- | :--- |
| Streamlit Community Cloud | free for public apps | deploy from a GitHub repository |
| Hugging Face Spaces | free tier | Streamlit, Gradio or Docker spaces |
| Docker container | your infrastructure | `streamlit run` behind a reverse proxy (M32) |

| Tool | Best for |
| :--- | :--- |
| Streamlit | fast data apps and prototypes from scripts |
| Dash (Plotly) | callback-based apps with fine control |
| Gradio | ML model demos (M30) |
| BI tools (Tableau, Power BI, Superset) | governed dashboards for many business users (Lessons 10–14) |

*Insight: Keep data apps small and testable: pre-aggregate the data, cache loading, put the logic in plain functions, and test the app with `AppTest` like any other code.*

## Real-World Use Case or Analogy

A Streamlit app is like a **pop-up shop**: quick to open, cheap to run, ideal to test an idea with real customers. When the idea works and many people depend on it, you move to a permanent store with stronger rules (a BI platform or a full web application).

Data scientists share model results with Streamlit apps, analysts build self-service tools for colleagues, and ML teams build internal labelling and evaluation tools with it.